In [1]:
import sys
from pathlib import Path

import numpy as np
import numpy.typing as npt

# Make notebook robust to different launch directories.
_candidates = [Path.cwd(), *Path.cwd().parents]
for _base in _candidates:
    if (_base / "src").exists() and (_base / "data").exists():
        if str(_base / "src") not in sys.path:
            sys.path.insert(0, str(_base / "src"))
        project_root = _base
        break
else:
    raise RuntimeError("Could not locate project root containing src/ and data/")

from model import square_lattice
from smatrix import L, coord_convert, create_self_energy_interpolator_numba

sigma_data = np.load(project_root / "data" / "sigma_grid0f6a.npz")
kx = sigma_data["kx"]
ky = sigma_data["ky"]
sigma_grid = sigma_data["sigma_grid"]
sigma_func_period_numba = create_self_energy_interpolator_numba(
    kx, ky, sigma_grid, lattice=square_lattice
)


def singlePhoton_G_filter(E: float, lattice) -> list[npt.NDArray[np.int_]]:
    """
    Return a sorted list of all the reciprocal lattice vector
    G = (m*q, n*q) labels an open single-photon Bragg channel, i.e. there exists
    some q_parallel in the first square Brillouin zone [-q/2, q/2]^2 satisfying
        |q_parallel + G| <= E.
    Assumes a square reciprocal lattice with spacing q = lattice.q.
    """
    int_list = []
    q = float(lattice.q)
    max_norm = E + np.sqrt(2) / 2 * q
    R = int(np.floor(max_norm / q))

    for m in range(-R, R + 1):
        for n in range(-R, R + 1):
            d_min_x = q * np.maximum(abs(m) - 1 / 2, 0.0)
            d_min_y = q * np.maximum(abs(n) - 1 / 2, 0.0)
            d_min = np.sqrt(d_min_x**2 + d_min_y**2)
            if d_min <= E:
                int_list.append(np.array([m, n]))
    int_list.sort(key=lambda x: (x[0], x[1]))
    result_list = [q * x for x in int_list]
    return result_list


def kr_delta(a: int, b: int):
    return 1 if a == b else 0

In [20]:
E = 200
filtered_G = singlePhoton_G_filter(E, square_lattice)
# Off-shell channels are expected; silence invalid-sqrt warning only here.
kG = np.array([50, 60]) + filtered_G
kG_norm = np.linalg.norm(kG, axis=1)
mask = kG_norm < E 
#    t = coord_convert(np.array([50, 60]) + filtered_G, E)
print(kG)
print(mask)
print(kG[mask])


[[-116.66666667 -106.66666667]
 [-116.66666667   60.        ]
 [-116.66666667  226.66666667]
 [  50.         -106.66666667]
 [  50.           60.        ]
 [  50.          226.66666667]
 [ 216.66666667 -106.66666667]
 [ 216.66666667   60.        ]
 [ 216.66666667  226.66666667]]
[ True  True False  True  True False False False False]
[[-116.66666667 -106.66666667]
 [-116.66666667   60.        ]
 [  50.         -106.66666667]
 [  50.           60.        ]]


In [21]:
def S_disc_Bragg(E, k_para,sigma_func_period,lattice):
    "Return a n by n matrix with fixed E and k_para, where n is the number of the relevant Bragg channels."
    filtered_G = singlePhoton_G_filter(E, lattice)

    kG = k_para + filtered_G  # in-plane momentum in all the relevant channels
    # Off-shell channels are expected; silence invalid sqrt warning locally.
    kG = np.array([50, 60]) + filtered_G
    kG_norm = np.linalg.norm(kG, axis=1)
    mask = kG_norm < E
    kG = kG[mask]
    n = len(kG)
    S_matrix = np.zeros((n, n), dtype=np.complex128)
    # Construct the matrix element by element
    for i in range(n):  # index for outgoing photon
        for j in range(n):  # index for incoming photon
            k_cart_out = coord_convert(kG[i], E)
            k_cart_in = coord_convert(kG[j], E)

            J_out = E / k_cart_out[2]
            J_in = E / k_cart_in[2]
            first_term = kr_delta(i, j)
            second_term = (
                -1j
                / lattice.a**2
                * np.sqrt(J_out)
                * np.conj(lattice.ge(k_cart_out))
                * L(kG[j], E, lattice, sigma_func_period, direction="in")
                * np.sqrt(J_in)
            )
            S_matrix[i, j] = first_term + second_term
    S_matrix = np.matrix(S_matrix)
    return S_matrix


In [28]:
S = S_disc_Bragg(350, np.array([square_lattice.q/2-1, square_lattice.q/2-1]), sigma_func_period_numba, square_lattice)

print(np.linalg.eigvals(S))
print(S.H @ S)

[1.        -1.11022302e-16j 0.99972067-1.89222254e-01j
 1.        -2.77555756e-17j 1.        -1.40512602e-16j
 1.        -2.08166817e-17j 1.        +1.38777878e-16j
 1.        +1.15359111e-16j 1.        -2.77555756e-16j
 1.        -1.66533454e-16j 1.        +7.28583860e-17j
 1.        +4.92924259e-17j 1.        -2.77555756e-17j
 1.        -2.22044605e-16j 1.        +1.38777878e-16j]
[[1.00215708+0.00000000e+00j 0.00209023+0.00000000e+00j
  0.00212517-1.73472348e-18j 0.00193769+1.73472348e-18j
  0.00193418+0.00000000e+00j 0.00200004+0.00000000e+00j
  0.00204979+0.00000000e+00j 0.00193316+0.00000000e+00j
  0.00193173+0.00000000e+00j 0.00197181+0.00000000e+00j
  0.00475602+0.00000000e+00j 0.00198165+0.00000000e+00j
  0.00196549+1.73472348e-18j 0.0022421 +3.46944695e-18j]
 [0.00209023+0.00000000e+00j 1.00202546+0.00000000e+00j
  0.00205932+0.00000000e+00j 0.00187765+0.00000000e+00j
  0.00187424-1.73472348e-18j 0.00193806-1.73472348e-18j
  0.00198627+0.00000000e+00j 0.00187325-1.73472348e-1

In [38]:
import os

import matplotlib.pyplot as plt
from joblib import Parallel, delayed

# ----- Sampling setup (similar to plot_disp.ipynb) -----
n_k = 1000  # number of k samples along each axis in 1st BZ
n_energy_points = 50
n_jobs = 6

k_axis = np.linspace(-square_lattice.q / 2, square_lattice.q / 2, n_k)
E_values = np.linspace(square_lattice.omega_e-50, square_lattice.omega_e+50, n_energy_points)  # adjust as needed

kx, ky = np.meshgrid(k_axis, k_axis, indexing="ij")
k_points = np.column_stack((kx.ravel(), ky.ravel()))


k_blocks = np.array_split(k_points, max(1, n_jobs * 4))


def _eigvals_block(E: float, k_block: np.ndarray) -> np.ndarray:
    eigvals_list = []
    for k_para in k_block:
        try:
            S = S_disc_Bragg(float(E), k_para, sigma_func_period_numba,square_lattice)
            if S.size > 0:
                eigvals_list.append(np.linalg.eigvals(S))
        except Exception:
            # Skip points/energies that fail (e.g., no open channels)
            continue

    if eigvals_list:
        return np.concatenate(eigvals_list)
    return np.array([], dtype=np.complex128)


# Compute eigenvalues over (E, kx, ky)
all_blocks = []
for E in E_values:
    eig_blocks = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(_eigvals_block)(E, k_block) for k_block in k_blocks
    )
    eig_blocks = [vals for vals in eig_blocks if vals.size > 0]
    if eig_blocks:
        all_blocks.append(np.concatenate(eig_blocks))

all_eigvals = (
    np.concatenate(all_blocks) if all_blocks else np.array([], dtype=np.complex128)
)

# ----- Plot on complex plane with unit circle -----
theta = np.linspace(0, 2 * np.pi, 800)
unit_circle = np.exp(1j * theta)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(unit_circle.real, unit_circle.imag, "k--", lw=1.5, label="|z| = 1")

if all_eigvals.size > 0:
    ax.scatter(
        all_eigvals.real,
        all_eigvals.imag,
        s=8,
        c="tab:blue",
        alpha=0.35,
        edgecolors="none",
        label=f"eigenvalues ({all_eigvals.size})",
    )

ax.axhline(0.0, color="gray", lw=0.8)
ax.axvline(0.0, color="gray", lw=0.8)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.set_title("Eigenvalues over 1st BZ and multiple energies")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

KeyboardInterrupt: 